Berikut panduan lengkap fine-tuning `intfloat/multilingual-e5-small` untuk domain hukum Bahasa Indonesia:

---

## Fine-Tuning `multilingual-e5-small` untuk Domain Hukum Indonesia

### 1. Persiapan Dataset

Ini adalah langkah terpenting. Kamu butuh pasangan kalimat hukum. Ada 3 format yang umum digunakan:

**Format A — (anchor, positive)** — paling mudah dibuat:
```
query: apa itu hak cipta?     → passage: hak cipta adalah hak eksklusif pencipta...
query: syarat sah perjanjian  → passage: perjanjian sah apabila memenuhi empat syarat...
```

**Format B — (anchor, positive, negative)** — lebih kuat:
```
anchor:   "query: apa itu wanprestasi?"
positive: "passage: wanprestasi adalah tidak terpenuhinya prestasi dalam perjanjian..."
negative: "passage: hak cipta diatur dalam UU No. 28 Tahun 2014..."
```

**Sumber data hukum yang bisa dipakai:**
- Putusan pengadilan dari SIPP/Mahkamah Agung
- Pasal-pasal KUHP, KUHPerdata, UU spesifik
- FAQ hukum dari LBH atau situs hukumonline
- Pasangan pertanyaan–jawaban dari konsultasi hukum

> ⚠️ **Penting:** Model E5 menggunakan prefix `"query: "` untuk pertanyaan/query dan `"passage: "` untuk dokumen/jawaban. Tetap gunakan prefix ini saat fine-tuning agar konsisten dengan pelatihan aslinya.

---

### 2. Instalasi

```bash
pip install sentence-transformers datasets transformers torch
```

---

### 3. Kode Fine-Tuning (Sentence Transformers v3)

Pendekatan yang direkomendasikan menggunakan `SentenceTransformerTrainer` dengan `MultipleNegativesRankingLoss` — loss function yang paling efektif untuk task embedding.

```python
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# ── 1. Load base model ──────────────────────────────────────────
model = SentenceTransformer("intfloat/multilingual-e5-small")

# ── 2. Siapkan dataset hukum ────────────────────────────────────
# Contoh format (anchor, positive) — minimal yang dibutuhkan
train_data = {
    "anchor": [
        "query: apa itu wanprestasi dalam hukum perdata?",
        "query: syarat sahnya suatu perjanjian",
        "query: pengertian hak tanggungan",
        "query: apa yang dimaksud delik aduan?",
        # ... minimal 500-1000 pasang untuk hasil yang baik
    ],
    "positive": [
        "passage: wanprestasi adalah keadaan dimana debitur tidak memenuhi kewajibannya yang diatur dalam perjanjian sesuai Pasal 1243 KUHPerdata.",
        "passage: berdasarkan Pasal 1320 KUHPerdata, syarat sah perjanjian meliputi kesepakatan, kecakapan, suatu hal tertentu, dan sebab yang halal.",
        "passage: hak tanggungan adalah hak jaminan yang dibebankan pada hak atas tanah sebagaimana diatur dalam UU No. 4 Tahun 1996.",
        "passage: delik aduan adalah tindak pidana yang hanya dapat dituntut apabila ada pengaduan dari pihak yang dirugikan.",
    ]
}

train_dataset = Dataset.from_dict(train_data)

# ── 3. Loss function ────────────────────────────────────────────
loss = MultipleNegativesRankingLoss(model)

# ── 4. Training arguments ───────────────────────────────────────
args = SentenceTransformerTrainingArguments(
    output_dir="e5-small-hukum-id",
    num_train_epochs=3,
    per_device_train_batch_size=32,   # Lebih besar = lebih baik untuk MNR loss
    learning_rate=2e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # Penting untuk MNR loss
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,  # Matikan jika tidak ada GPU
)

# ── 5. Train ────────────────────────────────────────────────────
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()
model.save_pretrained("e5-small-hukum-id-final")
```

---

### 4. Jika Punya Data Triplet (anchor, positive, negative)

```python
# Format triplet lebih kuat karena ada hard negative eksplisit
train_data = {
    "anchor":   ["query: apa itu wanprestasi?", ...],
    "positive": ["passage: wanprestasi adalah ...", ...],
    "negative": ["passage: hak cipta adalah ...", ...],  # dokumen tidak relevan
}
# Loss function sama: MultipleNegativesRankingLoss
```

---

### 5. Evaluasi Model

```python
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("e5-small-hukum-id-final")

query = "query: apa hukuman untuk kasus pencurian?"
passages = [
    "passage: Pasal 362 KUHP mengatur bahwa pencurian diancam pidana penjara paling lama 5 tahun.",
    "passage: hak tanggungan adalah hak jaminan atas tanah.",
    "passage: wanprestasi diatur dalam Pasal 1243 KUHPerdata.",
]

q_emb = model.encode(query)
p_embs = model.encode(passages)

scores = cos_sim(q_emb, p_embs)
for p, s in zip(passages, scores[0]):
    print(f"Score: {s:.4f} | {p[:60]}...")
```

---

### 6. Tips untuk Domain Hukum Indonesia

| Aspek | Rekomendasi |
|---|---|
| **Ukuran data** | Minimal 500 pasang; idealnya 2.000–10.000 |
| **Batch size** | Gunakan 32–64 (MNR loss butuh batch besar) |
| **Epoch** | 3–5 epoch, hindari overfitting |
| **Hard negatives** | Ambil pasangan pasal dari UU yang berbeda agar model bisa membedakan topik hukum |
| **Prefix** | Selalu pakai `"query: "` dan `"passage: "` |
| **Evaluasi** | Buat test set manual: 50–100 query + relevan/tidak relevan |

### 7. Alternatif: Jika Data Sangat Terbatas

Gunakan **`TripletLoss`** dengan **data sintetis** — generate pasangan query-passage menggunakan LLM (misal Claude/GPT) dari teks hukum yang kamu punya, lalu fine-tune dari sana.

---

Apakah kamu sudah punya dataset hukumnya, atau butuh bantuan membuat pipeline untuk generate data sintetis dari dokumen hukum yang ada?